# ⚔️ B1　Boss 戰：勇者咖啡 A/B 測試
**統計冒險之旅 2026**　｜　Day 1（09/21 一）🌄 統計之丘　｜　Boss 戰　｜　🏅 200 XP

📖 Day 1 綜合；資料：新舊菜單 A/B 測試


### 🎯 這一關你會學到
- 兩組比較的完整流程：敘述 → 視覺化 → 檢定 → 信賴區間 → 給老闆的結論

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
> 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "B1"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["B-1", "B-2", "B-3", "B-4", "B-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B_1(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "摘要"), 列=2, 含欄位=["count", "mean", "median", "std"])
    if not ok: return (False, msg)
    if not 約等於(抓變數(ns, "A平均"), 158.04, 0.15): return (False, "A平均 不對。")
    return (約等於(抓變數(ns, "B平均"), 173.40, 0.15), "B平均 = 摘要.loc['B 新菜單', 'mean']。")
任務定義("B-1", _check_B_1, 提示="agg([\"count\", \"mean\", \"median\", \"std\"])。")

def _check_B_2(run):
    out, ns = run()
    if len(run.figs) < 2: return (False, "要畫出兩張圖。")
    return ("客單價" in " ".join(f["title"] for f in run.figs), "標題要包含「客單價」。")
任務定義("B-2", _check_B_2, 提示="x='組別'；hue='組別'。")

def _check_B_3(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "差距"), 15.3583, 0.1): return (False, "差距 = B組.mean() - A組.mean()。")
    if not 約等於(抓變數(ns, "p值"), 0.004042, 0.002): return (False, "p值 不對：ttest_ind(B組, A組, equal_var=False)。")
    return (str(抓變數(ns, "結論")) == "顯著", "p < 0.05 → 顯著。")
任務定義("B-3", _check_B_3, 提示="B組 = ab[ab['組別'] == 'B 新菜單']['客單價']。")

def _check_B_4(run):
    out, ns = run()
    if not (約等於(抓變數(ns, "下界"), 5.091, 1.0) and 約等於(抓變數(ns, "上界"), 25.712, 1.0)): return (False, "np.percentile(差們, 2.5) 與 97.5；rng 種子 42、2000 次。")
    ok, msg = 資料框像(抓變數(ns, "加購交叉表"), 列=2, 欄=2)
    if not ok: return (False, msg)
    return (約等於(抓變數(ns, "加購p值"), 0.132006, 0.002), "加購p值 = chi2_contingency(加購交叉表)[1]（第二個回傳值）。")
任務定義("B-4", _check_B_4, 提示="chi2_contingency 回傳 (卡方, p, 自由度, 期望)，p 在索引 1。")

def _check_B_5(run):
    out, ns = run()
    d = 抓變數(ns, "結論", dict)
    for k in ["新菜單客單價提升", "客單價p值", "差距95%區間", "加購率變化", "加購p值", "建議"]:
        if k not in d: return (False, f"結論 缺少「{k}」。")
    if not 約等於(d["新菜單客單價提升"], 15.36, 0.2): return (False, "新菜單客單價提升 應該等於 差距。")
    if not 約等於(d["加購率變化"], 0.1000, 0.005): return (False, "加購率變化 = B 加購率 − A 加購率。")
    return (isinstance(d["建議"], str) and len(d["建議"]) >= 10, "建議 要是一句至少 10 個字的話。")
任務定義("B-5", _check_B_5, 提示="建議 是一個字串。")

## ⚔️ Boss 戰：勇者咖啡 A/B 測試——新菜單真的有效嗎？
老闆在**信義店**試賣新菜單：8/18–8/31 隨機把客人分成兩組——**A 組看舊菜單、B 組看新菜單**，各 120 位，記錄每位客人的**客單價**與**有沒有加購甜點**。
老闆的問題：「新菜單讓客人花更多錢嗎？加購率有變高嗎？要不要全面換新菜單？」

資料：`https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/coffee_ab_test.csv`（欄位：顧客編號、日期、組別、客單價、加購甜點）

> 這一關是 Day 1 的總驗收：敘述 → 視覺化 → 檢定 → 信賴區間 → 給老闆的結論。每一步都在前面的關卡出現過。

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt
ab = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/coffee_ab_test.csv")
ab.head()

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

### 🎯 任務 B-1　分組摘要

用 `groupby('組別')` 算出客單價的 `count、mean、median、std`（四捨五入 1 位）存成 `摘要`；再分別取出 `A平均` 與 `B平均`（`摘要.loc['A 舊菜單', 'mean']` …）。

In [ ]:
# 🎯 任務 B-1　分組摘要（請保留這一行）
摘要 = ab.groupby("組別")["客單價"].agg([???]).round(1)
A平均 = 摘要.loc["A 舊菜單", "mean"]
B平均 = ???
print(摘要)
print("差距", round(B平均 - A平均, 1))

In [ ]:
檢查("B-1")   # ◀ 執行這一格，看看任務 B-1 有沒有過關

### 🎯 任務 B-2　視覺化比較

畫兩張圖：(1) 以 `組別` 分組的客單價**盒鬚圖**；(2) 兩組客單價疊在一起的**直方圖**（`sns.histplot(data=ab, x='客單價', hue='組別')`）。至少一張圖的標題要包含「客單價」。

In [ ]:
# 🎯 任務 B-2　視覺化比較（請保留這一行）
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=ab, x=???, y="客單價", ax=ax[0]); ax[0].set_title("兩組客單價的盒鬚圖")
sns.histplot(data=ab, x="客單價", hue=???, ax=ax[1]); ax[1].set_title("兩組客單價的分布")
plt.tight_layout(); plt.show()

In [ ]:
檢查("B-2")   # ◀ 執行這一格，看看任務 B-2 有沒有過關

### 🎯 任務 B-3　t 檢定：差距是真的嗎？

把兩組客單價分別存成 `A組`、`B組`（Series），做 Welch t 檢定存成 `t值`、`p值`，算出 `差距`（B 平均 − A 平均），`結論` 設成 `"顯著"` 或 `"不顯著"`。

In [ ]:
# 🎯 任務 B-3　t 檢定：差距是真的嗎？（請保留這一行）
A組 = ab[ab["組別"] == "A 舊菜單"]["客單價"]
B組 = ???
t值, p值 = stats.ttest_ind(B組, A組, equal_var=False)
差距 = ???
結論 = ???
print(round(差距, 1), round(t值, 2), round(p值, 4), 結論)

In [ ]:
檢查("B-3")   # ◀ 執行這一格，看看任務 B-3 有沒有過關

### 🎯 任務 B-4　差距的信賴區間與加購率

用 bootstrap（種子 42、2,000 次、每次分別抽後放回再相減）算出差距的 95% 區間 `下界`、`上界`；再做 `組別 × 加購甜點` 的交叉表 `加購交叉表`，用卡方檢定算出 `加購p值`。

In [ ]:
# 🎯 任務 B-4　差距的信賴區間與加購率（請保留這一行）
rng = np.random.default_rng(42)
差們 = [rng.choice(B組.values, len(B組), replace=True).mean() - rng.choice(A組.values, len(A組), replace=True).mean() for _ in range(2000)]
下界 = ???
上界 = ???
加購交叉表 = pd.crosstab(ab["組別"], ab["加購甜點"])
加購p值 = stats.chi2_contingency(加購交叉表)[???]
print("差距 95% 區間：", round(下界, 1), "~", round(上界, 1))
print(加購交叉表)
print("加購率 A", round(A組.size and ab[ab["組別"] == "A 舊菜單"]["加購甜點"].mean(), 3), "B", round(ab[ab["組別"] == "B 新菜單"]["加購甜點"].mean(), 3), "p =", round(加購p值, 3))

In [ ]:
檢查("B-4")   # ◀ 執行這一格，看看任務 B-4 有沒有過關

### 🎯 任務 B-5　給老闆的結論

把結果填進字典 `結論`（從前面的變數取值，不要手打數字），並用迴圈印出。`建議` 請寫一句話（字串，至少 10 個字），例如「客單價顯著提升，建議全面換新菜單，加購率再觀察」。

In [ ]:
# 🎯 任務 B-5　給老闆的結論（請保留這一行）
結論 = {
    "新菜單客單價提升": round(差距, 1),
    "客單價p值": round(p值, 4),
    "差距95%區間": (round(下界, 1), round(上界, 1)),
    "加購率變化": round(ab[ab["組別"] == "B 新菜單"]["加購甜點"].mean() - ab[ab["組別"] == "A 舊菜單"]["加購甜點"].mean(), 3),
    "加購p值": round(加購p值, 3),
    "建議": ???,
}
for k, v in 結論.items():
    print(f"{k}：{v}")

In [ ]:
檢查("B-5")   # ◀ 執行這一格，看看任務 B-5 有沒有過關

## 🎤 成果分享（3 分鐘）
用你的結論，向「老闆」報告：新菜單值得全面上線嗎？哪個數字最有說服力？哪個數字還要再等等？

## 🌟 進階挑戰（不計分）
1. 如果只看前 60 位客人，t 檢定的 p 值會變成多少？樣本變小，結論會變嗎？
2. 用 `sns.pointplot` 畫出兩組客單價的平均與信賴區間。

---
## 🔑 通關密語
　Day 1 通關！你已經會把「差距」變成「證據」。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📈 L04 線性迴歸** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0/notebooks/L04_linear_regression.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/